# Week 5: Height PRS for 1000 Genomes and a 23andMe Sample

## This week at a glance

**Big idea:** This week I build a real polygenic risk score, for height, using GWAS summary statistics from a large study. I score all 489 people in 1000 Genomes to get the population's bell curve, then score Tiffany's 23andMe data and see where she lands. Since I know her real height (5'6"), I can check if her score makes sense.

**What I cover**
- What GWAS summary statistics look like, and how to clean them (QC)
- Lining up alleles between the GWAS, 1000 Genomes, and 23andMe
- P+T genome-wide at a fixed threshold, and why it's fixed this time
- Turning a score into a percentile, and what it does and doesn't say about one person

**My readings**
No new reading this week. Optional background: Amariuta 2020 (*Nature Genetics*), Martin 2019 and Khera 2018 (both from Week 1).

**My tasks**
1. Convert the 23andMe data to VCF and PLINK formats (bcftools tutorial).
2. Prepare the height PRS: P+T on the height GWAS at a fixed p-value threshold of 10 x 10⁻⁴ (= 0.001).
3. Generate the PRS for each 1000 Genomes person and for Tiffany.
4. Plot the 1000 Genomes height PRS distribution and report where Tiffany falls.

**Weekly questions I answer on Gradescope**
1. Given that Tiffany is 5'6", is her height PRS sensible with respect to the population?
2. Upload an image of the 1000 Genomes height PRS bell curve with a line marking Tiffany's PRS.

**My data**
- Height GWAS summary statistics from the PRS tutorial (`../data/raw/Height.gwas.txt.gz`)
- Genotypes: 1000 Genomes European samples (`../data/raw/LDREF/`)
- Tiffany's 23andMe raw data (`../data/raw/tiffany_23andme.txt`)

## Biology background and vocabulary

### GWAS summary statistics

A GWAS tests every SNP against a trait in a big group of people, the same regression I ran for eQTLs, but with height instead of expression. Researchers usually share only the per-SNP results, not the people's genotypes (Week 2: raw genotypes are identifiable). These results are the **summary statistics**.

| Column | Meaning |
|---|---|
| `CHR`, `BP` | Chromosome and position |
| `SNP` | rsID |
| `A1` | The **effect allele**: the effect size is per copy of this allele |
| `A2` | The other allele |
| `N` | How many people were in the GWAS |
| `OR`, `SE`, `P` | Effect size (as an odds ratio), its standard error, and p-value |
| `INFO` | How confident the genotypes were (imputation quality, 0 to 1) |
| `MAF` | Minor allele frequency in the GWAS |

The PRS formula needs an additive effect, so I convert the odds ratio to a **beta**: **beta = log(OR)**. OR above 1 gives a positive beta (taller), OR below 1 a negative beta (shorter).

### Cleaning the summary statistics (QC)

| Filter | Why |
|---|---|
| INFO > 0.8 | Drop SNPs whose genotypes were poorly guessed |
| MAF > 0.01 | Drop rare SNPs, whose effect sizes are noisy |
| Drop A/T and C/G SNPs | These are **ambiguous** (see below) |
| Drop duplicate rsIDs | Can't tell which row is right |

**Why A/T and C/G are ambiguous.** From Week 1, DNA has two strands, and A pairs with T, C with G. Different datasets sometimes read the opposite strand. For an A/G SNP, the other strand reads T/C, so I can always tell when a dataset flipped. But for an A/T SNP, the other strand also reads T/A, so a flip is invisible:

```
SNP type   top strand   other strand   can I tell if it flipped?
A/G        A / G        T / C          yes, the letters change
A/T        A / T        T / A          no, same two letters
```

If I can't tell, I might count the wrong allele and flip the sign of its effect, so those SNPs are dropped.

### Lining up alleles

Every dataset has to count the **same** allele, or a SNP's effect points the wrong way. The `.bed` file counts the `.bim` file's first allele. So for each SNP I rewrite the GWAS effect as "per copy of the 1000 Genomes counted allele":

| GWAS (A1/A2) | 1000 Genomes counted / other | What I do |
|---|---|---|
| A / G | A / G | Same, keep beta |
| G / A | A / G | Swapped, use **-beta** |
| T / C | A / G | Other strand (T=A, C=G), keep beta |
| A / C | A / G | Doesn't match at all, drop the SNP |

### Why the threshold is fixed this time

In Week 4 I tried 5 thresholds and let the tuning group pick, because I had real expression values to check against. 1000 Genomes has no height measurements, so there's nothing to tune on. I use a fixed p < 0.001 (the task's 10 x 10⁻⁴).

### From a score to a percentile

A PRS by itself has no units that mean anything. What matters is where a person sits compared to others, so I score all 489 1000 Genomes people and compare.

- **Percentile:** the share of the panel with a lower score. 80th percentile = higher than 80% of people.
- **z-score:** how many standard deviations above (+) or below (-) the average. On a bell curve, z = 1 is about the 84th percentile, z = 2 about the 98th.

### What a height PRS can and can't say

- Height is about 80% heritable, but a P+T score from one GWAS captures only part of that. So a PRS **shifts the odds** of being tall or short. It doesn't decide a person's height.
- The score doesn't know someone's **sex**, and men average about 5 inches taller. So a PRS percentile should be compared to height **within** the same sex.
- It works best for people of the same ancestry as the GWAS (Martin 2019). The panel is European, so a person of other ancestry may land in a misleading spot.

### 23andMe raw data and file formats

A 23andMe download is a text file with one row per SNP on their chip (about 600,000):

```
# rsid      chromosome   position   genotype
rs4477212   1            82154      AA
rs3094315   1            752566     AG
rs12562034  1            768448     --        <- "--" means the call failed
```

| Format | What it is |
|---|---|
| **23andMe .txt** | Letters per SNP for one person |
| **VCF** | The standard text format for genotypes. One row per variant, one column per person |
| **PLINK (.bed/.bim/.fam)** | The compact format I've used all quarter |

The task follows the bcftools tutorial to go 23andMe -> VCF -> PLINK:

```bash
# 23andMe -> VCF (needs the human reference genome, build 37, to know which letter is the reference)
bcftools convert --tsv2vcf tiffany_23andme.txt -f human_g1k_v37.fasta -s Tiffany -Oz -o tiffany.vcf.gz
# VCF -> PLINK
plink2 --vcf tiffany.vcf.gz --make-bed --out tiffany
```

I don't have the reference genome file (about 3 GB), so in Step 7 I build the PLINK files directly in Python, coded the same way as 1000 Genomes so they can be merged. The end result is the same `.bed/.bim/.fam` set.

### Week 5 terms

| Term | Meaning |
|---|---|
| **Summary statistics** | Per-SNP GWAS results (effect, p-value) without anyone's raw genotypes |
| **Effect allele (A1)** | The allele the effect size is counted for |
| **Odds ratio (OR) / beta** | Effect size. beta = log(OR) |
| **INFO score** | Imputation quality, 0 to 1 |
| **Strand-ambiguous SNP** | A/T or C/G SNP, where a strand flip can't be detected |
| **Allele alignment** | Making every dataset count the same allele |
| **Reference panel** | 1000 Genomes: used for LD (clumping) and as the comparison population |
| **Percentile / z-score** | Where one person sits in the panel's distribution |
| **VCF** | Standard text format for genotypes |
| **bcftools** | Tool for working with VCF files |

## Task 2: Prepare the height PRS

### Step 1: Look at the height summary statistics

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from bed_reader import open_bed, to_bed
from scipy import stats

genotype_folder = "../data/raw/LDREF"
gwas_path = "../data/raw/Height.gwas.txt.gz"
tiffany_path = "../data/raw/tiffany_23andme.txt"

height_gwas = pd.read_csv(gwas_path, sep="\t")

print("SNPs in the GWAS:", len(height_gwas))
print("People in the GWAS (N):", height_gwas["N"].iloc[0])
height_gwas.head()

**What I see:** I see 529,504 SNPs from a GWAS of about 388,000 people, with one row per SNP and the columns described above. The effect is stored as an odds ratio (`OR`), with values very close to 1, since each SNP has a tiny effect on height.

### Step 2: QC the summary statistics

In [ ]:
ambiguous_pairs = {("A", "T"), ("T", "A"), ("C", "G"), ("G", "C")}

passes_info_and_maf = (height_gwas["INFO"] > 0.8) & (height_gwas["MAF"] > 0.01)
clean_gwas = height_gwas[passes_info_and_maf].copy()
clean_gwas["A1"] = clean_gwas["A1"].str.upper()
clean_gwas["A2"] = clean_gwas["A2"].str.upper()

is_ambiguous = np.array([(a1, a2) in ambiguous_pairs for a1, a2 in zip(clean_gwas["A1"], clean_gwas["A2"])])
number_ambiguous = int(is_ambiguous.sum())
clean_gwas = clean_gwas[~is_ambiguous]

is_duplicate = clean_gwas["SNP"].duplicated(keep=False)
clean_gwas = clean_gwas[~is_duplicate].reset_index(drop=True)

clean_gwas["BETA"] = np.log(clean_gwas["OR"])

print("Start:", len(height_gwas))
print("Removed by INFO or MAF:", int((~passes_info_and_maf).sum()))
print("Removed as strand-ambiguous:", number_ambiguous)
print("Removed as duplicates:", int(is_duplicate.sum()))
print("SNPs after QC:", len(clean_gwas))

**What I see:** I see the QC keeps 499,617 of the 529,504 SNPs (94%). Almost all of the removed SNPs (29,878) were strand-ambiguous A/T or C/G SNPs. Only 9 failed the INFO or MAF filters and there were no duplicates, so the file was already cleaned for quality. Every remaining SNP now has a `BETA`, its effect per copy of `A1`.

### Step 3: Match the GWAS SNPs to 1000 Genomes and line up alleles

In [ ]:
complement = str.maketrans("ACGT", "TGCA")
bim_column_names = ["chromosome", "snp_id", "genetic_distance", "position", "counted_allele", "other_allele"]

def align_to_1000_genomes(gwas_rows, snp_table):
    """Keep GWAS SNPs found in 1000 Genomes, and rewrite beta as the effect per copy of the 1000 Genomes counted allele."""
    merged = gwas_rows.merge(snp_table, left_on="SNP", right_on="snp_id", how="inner")
    gwas_a1, gwas_a2 = merged["A1"], merged["A2"]
    flipped_a1, flipped_a2 = gwas_a1.str.translate(complement), gwas_a2.str.translate(complement)
    counted, other = merged["counted_allele"], merged["other_allele"]

    same = ((gwas_a1 == counted) & (gwas_a2 == other)) | ((flipped_a1 == counted) & (flipped_a2 == other))
    swapped = ((gwas_a1 == other) & (gwas_a2 == counted)) | ((flipped_a1 == other) & (flipped_a2 == counted))

    merged = merged[same | swapped].copy()
    merged["beta_counted"] = np.where(same[same | swapped], merged["BETA"], -merged["BETA"])
    merged["was_swapped"] = swapped[same | swapped].to_numpy()
    return merged


snp_tables = {}
aligned_parts = []
for chromosome in range(1, 23):
    snp_table = pd.read_csv(f"{genotype_folder}/1000G.EUR.{chromosome}.bim", sep=r"\s+", header=None, names=bim_column_names)
    snp_table["column"] = np.arange(len(snp_table))
    snp_tables[chromosome] = snp_table
    aligned_parts.append(align_to_1000_genomes(clean_gwas[clean_gwas["CHR"] == chromosome], snp_table))

aligned_gwas = pd.concat(aligned_parts, ignore_index=True)

print("1000 Genomes SNPs:", sum(len(table) for table in snp_tables.values()))
print("GWAS SNPs found in 1000 Genomes with matching alleles:", len(aligned_gwas))
print("Of those, alleles listed in swapped order:", int(aligned_gwas["was_swapped"].sum()))
aligned_gwas[["SNP", "A1", "A2", "counted_allele", "other_allele", "BETA", "beta_counted", "P"]].head()

**What I see:** I see 138,480 GWAS SNPs are also in 1000 Genomes with matching alleles. That's only about a quarter of the cleaned GWAS, because LDREF has only about 1.2 million SNPs. 2,083 of them list the alleles in the opposite order. For those SNPs the two datasets list the alleles in opposite order, `beta_counted` is the negative of `BETA`. Now every effect is per copy of the allele the `.bed` file counts.

### Step 4: Threshold and clump (P+T)

I keep SNPs with p < 0.001, then clump them (r² > 0.1 within 250 kb), with LD measured in the 489 1000 Genomes people. I only need to clump the SNPs below the threshold: whether a SNP survives clumping depends only on SNPs with smaller p-values, which are also below the threshold.

In [ ]:
p_threshold = 0.001
clump_window = 250_000
clump_ld_cutoff = 0.1

def clump(snp_p_values, snp_positions, snp_genotypes):
    """Greedy clumping: keep the smallest p-value, drop SNPs within 250 kb with LD r² > 0.1, repeat."""
    standardized = snp_genotypes - np.nanmean(snp_genotypes, axis=0)
    standardized = np.nan_to_num(standardized)
    lengths = np.linalg.norm(standardized, axis=0)
    lengths[lengths == 0] = np.inf
    standardized = standardized / lengths

    still_available = np.ones(len(snp_p_values), dtype=bool)
    kept = np.zeros(len(snp_p_values), dtype=bool)
    for snp in np.argsort(snp_p_values, kind="stable"):
        if not still_available[snp]:
            continue
        kept[snp] = True
        still_available[snp] = False
        nearby = np.flatnonzero(still_available & (np.abs(snp_positions - snp_positions[snp]) <= clump_window))
        ld_r_squared = (standardized[:, nearby].T @ standardized[:, snp]) ** 2
        still_available[nearby[ld_r_squared > clump_ld_cutoff]] = False
    return kept


def select_score_snps(aligned_rows):
    """P+T across all chromosomes. Returns the SNPs that go into the score."""
    selected_parts = []
    for chromosome in range(1, 23):
        candidates = aligned_rows[(aligned_rows["CHR"] == chromosome) & (aligned_rows["P"] < p_threshold)]
        candidates = candidates.sort_values("column").reset_index(drop=True)
        if candidates.empty:
            continue
        with open_bed(f"{genotype_folder}/1000G.EUR.{chromosome}.bed") as bed_file:
            candidate_genotypes = bed_file.read(index=np.s_[:, candidates["column"].to_numpy()], dtype="float64")
        kept = clump(candidates["P"].to_numpy(), candidates["position"].to_numpy(), candidate_genotypes)
        selected_parts.append(candidates[kept])
    return pd.concat(selected_parts, ignore_index=True)


below_threshold = aligned_gwas["P"] < p_threshold
score_snps = select_score_snps(aligned_gwas)

print("Aligned SNPs with p < 0.001:", int(below_threshold.sum()))
print("SNPs left after clumping (in the score):", len(score_snps))
score_snps.groupby("CHR").size().plot(kind="bar", color="steelblue", figsize=(9, 3), rot=0)
plt.xlabel("Chromosome")
plt.ylabel("SNPs in the score")
plt.title("Height PRS SNPs per chromosome")
plt.show()

**What I see:** I see 17,151 SNPs pass p < 0.001, and clumping cuts that to 8,675 roughly independent SNPs, so about half were removed as LD copies of a stronger nearby SNP. The bar chart shows SNPs on every chromosome, from about 720 on chromosome 1 down to about 120 on chromosome 21. Bigger chromosomes generally have more. That's what a highly polygenic trait like height looks like: thousands of small effects everywhere in the genome, not a few big genes.

## Task 3: Generate the PRS for every 1000 Genomes person

### Step 5: Score the 489 people

In [ ]:
fam_column_names = ["family_id", "person_id", "dad_id", "mom_id", "sex", "trait_value"]
panel_people = pd.read_csv(f"{genotype_folder}/1000G.EUR.1.fam", sep=r"\s+", header=None, names=fam_column_names)["person_id"]

def score_1000_genomes(snps_in_score):
    """PRS = sum over score SNPs of (copies of the counted allele x beta). Missing calls count as the SNP's average."""
    total_score = np.zeros(len(panel_people))
    for chromosome, chromosome_snps in snps_in_score.groupby("CHR"):
        chromosome_snps = chromosome_snps.sort_values("column")
        with open_bed(f"{genotype_folder}/1000G.EUR.{chromosome}.bed") as bed_file:
            copies = bed_file.read(index=np.s_[:, chromosome_snps["column"].to_numpy()], dtype="float64")
        copies = np.where(np.isnan(copies), np.nanmean(copies, axis=0), copies)
        total_score += copies @ chromosome_snps["beta_counted"].to_numpy()
    return pd.Series(total_score, index=panel_people, name="height_prs")


panel_scores = score_1000_genomes(score_snps)

print("People scored:", len(panel_scores))
print("Average PRS:", round(panel_scores.mean(), 3))
print("Standard deviation:", round(panel_scores.std(), 3))
print("Lowest and highest:", round(panel_scores.min(), 2), round(panel_scores.max(), 2))

**What I see:** All 489 people get a score. The average is about 0.62, the standard deviation about 0.87, and scores range from about -2.8 to 3.1. The numbers themselves don't have units like inches, so what matters is how people compare to each other.

### Step 6: The 1000 Genomes height PRS distribution

In [ ]:
def plot_distribution(scores, person_score=None, person_label=None, person_percentile=None):
    mean, standard_deviation = scores.mean(), scores.std()
    x_values = np.linspace(scores.min() - standard_deviation, scores.max() + standard_deviation, 300)
    if person_score is not None:
        x_values = np.linspace(min(scores.min(), person_score) - standard_deviation,
                               max(scores.max(), person_score) + standard_deviation, 300)

    plt.figure(figsize=(8, 4.5))
    plt.hist(scores, bins=30, density=True, edgecolor="white",
             label=f"1000 Genomes Europeans (n={len(scores)})")
    plt.plot(x_values, stats.norm.pdf(x_values, mean, standard_deviation), linewidth=2, label="Bell curve fit")
    if person_score is not None:
        plt.axvline(person_score, color="C1", linewidth=2.5,
                    label=f"{person_label}: {person_percentile:.0f}th percentile")
    plt.xlabel("Height polygenic risk score")
    plt.ylabel("Density")
    plt.title("Height PRS across 1000 Genomes Europeans")
    plt.legend(loc="upper left", fontsize=9)
    plt.show()


plot_distribution(panel_scores)
print("Normality check (Shapiro-Wilk p-value):", round(stats.shapiro(panel_scores).pvalue, 3))

**What I see:** The scores form a bell curve centered around 0.6, and the fitted normal curve follows the bars closely. That's expected from Week 1: each person's score adds up thousands of small random pushes up and down, which mostly cancel out, so most people land in the middle and few land at the extremes.

It isn't a perfect bell curve, though. The middle is a little taller than the fitted curve, and the Shapiro-Wilk test gives p = 0.016, so the scores aren't exactly normal. With 489 people, this test catches even small departures. It doesn't affect Tiffany's percentile, because I calculate that from her actual rank among the 489 people, not from the fitted curve.

## Tasks 1 and 3: Tiffany's 23andMe data

### Step 7: Read the 23andMe file and convert it to PLINK

This cell only runs once `tiffany_23andme.txt` is in `data/raw`.

In [ ]:
have_tiffany_file = os.path.exists(tiffany_path)
print("Tiffany's 23andMe file found:", have_tiffany_file)

if have_tiffany_file:
    tiffany_calls = pd.read_csv(tiffany_path, sep="\t", comment="#", header=None,
                                names=["rsid", "chip_chromosome", "chip_position", "genotype"],
                                dtype={"rsid": str, "chip_chromosome": str, "genotype": str})
    tiffany_calls["genotype"] = tiffany_calls["genotype"].fillna("--").str.upper()

    print("SNPs on Tiffany's chip:", len(tiffany_calls))
    print("Failed calls (--):", int((tiffany_calls["genotype"] == "--").sum()))
    print("Chromosomes:", tiffany_calls["chip_chromosome"].unique().tolist())
    # Raw genotype rows are not printed, so saved outputs never publish personal genetic data

**What I see:** I see Tiffany's chip has 643,535 SNPs on chromosomes 1 to 22 plus X, Y, and MT (mitochondrial DNA). 13,451 calls failed (`--`), about 2%. Positions are on build 37, the same build as 1000 Genomes, so the two line up.

In [ ]:
def copies_of_counted_allele(genotype_letters, counted, other):
    """Turn a 23andMe genotype like 'AG' into 0, 1, or 2 copies of the 1000 Genomes counted allele."""
    if len(genotype_letters) != 2 or not set(genotype_letters) <= set("ACGT"):
        return np.nan                                  # failed call, indel, or single-letter call
    letters = list(genotype_letters)
    if not set(letters) <= {counted, other}:
        letters = [letter.translate(complement) for letter in letters]   # try the other strand
        if not set(letters) <= {counted, other}:
            return np.nan                              # letters don't match this SNP
    return float(sum(letter == counted for letter in letters))


if have_tiffany_file:
    all_panel_snps = pd.concat([table.assign(CHR=chromosome) for chromosome, table in snp_tables.items()], ignore_index=True)
    tiffany_matched = tiffany_calls.merge(all_panel_snps, left_on="rsid", right_on="snp_id", how="inner")
    tiffany_matched["copies"] = [copies_of_counted_allele(g, c, o) for g, c, o in
                                 zip(tiffany_matched["genotype"], tiffany_matched["counted_allele"], tiffany_matched["other_allele"])]
    tiffany_matched = tiffany_matched.dropna(subset=["copies"]).sort_values(["CHR", "position"]).reset_index(drop=True)

    # Save as PLINK files coded the same way as 1000 Genomes (Task 1)
    os.makedirs("../data/processed", exist_ok=True)
    to_bed("../data/processed/tiffany.bed",
           tiffany_matched["copies"].to_numpy(dtype="float32").reshape(1, -1),
           properties={"fid": ["Tiffany"], "iid": ["Tiffany"],
                       "chromosome": tiffany_matched["CHR"].astype(str).tolist(),
                       "sid": tiffany_matched["snp_id"].tolist(),
                       "cm_position": tiffany_matched["genetic_distance"].tolist(),
                       "bp_position": tiffany_matched["position"].tolist(),
                       "allele_1": tiffany_matched["counted_allele"].tolist(),
                       "allele_2": tiffany_matched["other_allele"].tolist()})

    print("Tiffany's SNPs that are also in 1000 Genomes with usable calls:", len(tiffany_matched))
    print("Saved ../data/processed/tiffany.bed/.bim/.fam")

**What I see:** I see 168,226 of her SNPs are also in 1000 Genomes with a usable call that matches the alleles. That's only about a quarter of her chip and about 14% of LDREF, because a 23andMe chip and the HapMap3 SNP set were designed separately. I saved those 168,226 calls as `tiffany.bed/.bim/.fam` in `data/processed`, using the same counted allele as 1000 Genomes.

### Step 8: Score Tiffany and the panel on the same SNPs

Tiffany's chip doesn't have every SNP in my 8,675-SNP score. If I score her on fewer SNPs than the 1000 Genomes people, her score would be shifted compared to them, and the comparison would be unfair. So I rebuild the score using only SNPs that **she** has, then rescore everyone with that same set.

In [ ]:
if have_tiffany_file:
    tiffany_snp_set = set(tiffany_matched["snp_id"])
    shared_score_snps = select_score_snps(aligned_gwas[aligned_gwas["SNP"].isin(tiffany_snp_set)])

    shared_panel_scores = score_1000_genomes(shared_score_snps)

    tiffany_copies = tiffany_matched.set_index("snp_id").loc[shared_score_snps["SNP"], "copies"].to_numpy()
    tiffany_score = float(tiffany_copies @ shared_score_snps["beta_counted"].to_numpy())

    tiffany_percentile = 100 * (shared_panel_scores < tiffany_score).mean()
    tiffany_z = (tiffany_score - shared_panel_scores.mean()) / shared_panel_scores.std()

    print("SNPs in the shared score:", len(shared_score_snps), "(vs", len(score_snps), "in the full score)")
    print("Correlation between full and shared panel scores:", round(np.corrcoef(panel_scores, shared_panel_scores)[0, 1], 3))
    print("Tiffany's PRS:", round(tiffany_score, 3))
    print(f"Tiffany's percentile: {tiffany_percentile:.0f}")
    print(f"Tiffany's z-score: {tiffany_z:.2f}")

**What I see:** Only 2,061 of the 8,675 SNPs in the full score are on her chip. Rebuilding the score from scratch on her SNPs gives 3,060 SNPs, since clumping can now keep SNPs that were removed before in favor of a neighbor she doesn't have. Across the 489 people, this shared score correlates 0.71 with the full score, so it's a noisier version of the same score.

Tiffany's PRS is 0.288, which puts her at the **50th percentile** (z = 0.05). Her genetic height score is right in the middle of the 1000 Genomes Europeans.

## Task 4 and Gradescope question 2: Tiffany on the bell curve

In [ ]:
if have_tiffany_file:
    plot_distribution(shared_panel_scores, tiffany_score, "Tiffany", tiffany_percentile)
else:
    print("Add tiffany_23andme.txt to data/raw and rerun from Step 7 to draw Tiffany's line.")

**What I see:** The orange line sits right at the peak of the bell curve, so about half of the 1000 Genomes people have a higher height score than Tiffany and half have a lower one. This plot is the image for Gradescope question 2. On a Mac, right-click the plot in Jupyter and save it, or take a screenshot.

### Step 9: Does her position depend on the threshold?

The p < 0.001 cutoff was fixed, not tuned. To check that Tiffany's spot isn't just a result of that one choice, I redo the shared score at a few other thresholds and see where she lands each time.

In [ ]:
if have_tiffany_file:
    tiffany_snps = aligned_gwas[aligned_gwas["SNP"].isin(tiffany_snp_set)]
    tiffany_copies_by_snp = tiffany_matched.set_index("snp_id")["copies"]
    threshold_rows = []
    for threshold in [5e-8, 1e-5, 1e-3, 1e-2, 5e-2]:
        p_threshold = threshold
        snps_at_threshold = select_score_snps(tiffany_snps)
        scores_at_threshold = score_1000_genomes(snps_at_threshold)
        score = float(tiffany_copies_by_snp.loc[snps_at_threshold["SNP"]].to_numpy() @ snps_at_threshold["beta_counted"].to_numpy())
        threshold_rows.append({"p_threshold": threshold, "snps_in_score": len(snps_at_threshold),
                               "tiffany_percentile": round(100 * (scores_at_threshold < score).mean()),
                               "tiffany_z": round((score - scores_at_threshold.mean()) / scores_at_threshold.std(), 2)})
    p_threshold = 0.001   # back to the main threshold
    display(pd.DataFrame(threshold_rows))

**What I see:** Tiffany lands between the 45th and 72nd percentile at every threshold, with 498 SNPs at the strictest (5e-8) and 9,051 at the loosest (0.05). Her exact percentile moves around, which is expected when the score changes by thousands of SNPs, but she's never in either tail. Most thresholds put her a little above the middle.

## Week 5 answers

### Task questions

**1. Convert the 23andMe data to VCF and PLINK.**
The bcftools route (`bcftools convert --tsv2vcf`, then `plink2 --vcf`) needs the build 37 reference genome. I built the PLINK files (`data/processed/tiffany.bed/.bim/.fam`) directly in Python instead, coded with the same counted allele as 1000 Genomes so the two can be merged or scored together.

**2. Prepare the height PRS.**
QC (INFO > 0.8, MAF > 0.01, no ambiguous or duplicate SNPs) kept 499,617 of 529,504 GWAS SNPs. 138,480 matched 1000 Genomes. 17,151 had p < 0.001, and clumping (r² > 0.1, 250 kb) left 8,675 SNPs in the score.

**3. Generate the PRS.**
All 489 1000 Genomes people were scored (average 0.62, standard deviation 0.87). Only 2,061 of the 8,675 score SNPs are on Tiffany's chip, so I rebuilt the score using only SNPs she has (3,060 SNPs after clumping) and rescored the panel on that same set. Her score is 0.288.

**4. Distribution and Tiffany's relative risk.**
The 1000 Genomes scores form a bell curve. Tiffany's score is at the 50th percentile (z = 0.05), so her genetic predisposition for height is average compared to these Europeans. Across thresholds from 5e-8 to 0.05 she stays between the 45th and 72nd percentile, so she's never near either tail.

### Gradescope questions

**1. Given that Tiffany is 5'6", is her height PRS sensible with respect to the population?**
Yes, it's reasonable. Her height PRS is at the 50th percentile of the 1000 Genomes Europeans (z = 0.05), which predicts about average height for a woman. At 5'6" she's a little above average for a woman (US women average about 5'4"), so she's slightly taller than her score alone would suggest, but not by an amount that contradicts it. A height PRS built this way explains only part of the differences in height between people, so a gap of an inch or two for one person is normal. Her position is also stable: across p-value thresholds from 5e-8 to 0.05 she stays between the 45th and 72nd percentile, never in either tail, and most thresholds put her a bit above the middle, which matches her real height.

Two cautions. First, the score has no term for sex, and the panel mixes men and women, so her percentile is a percentile of genetic score, not of height among women. Second, only 3,060 SNPs could be used because her chip and the 1000 Genomes SNP set overlap poorly, so her score is a noisier version of the full 8,675-SNP score (they correlate 0.71 across the panel). One person can't validate or disprove a PRS, but her result is consistent with her height.

**2. Upload an image of the 1000 Genomes height PRS bell curve with a line marking Tiffany's PRS.**
The plot from the last code cell.